# canvas-host: simple WebGL2 example

This example shows the minimal worker-side setup:

1. attach an `OffscreenCanvas` and transfer it to the kernel worker
2. create a WebGL2 context
3. start the normalized input event queue
4. poll and log pointer and keyboard events in a render loop
5. stop the loop and release the canvas

The shader draws a glow under the pointer. Clicking the canvas or holding a key
changes its color. All cells are C++ and use `<canvas_host/canvas_host.hpp>`.


In [ ]:
#include <canvas_host/canvas_host.hpp>

#include <iostream>

canvas_host::WorkerConfig config;
config.width = 640;
config.height = 400;
config.status_text = "canvas-host / simple WebGL2";

auto canvas = canvas_host::attach_worker(config);

std::cout << "canvas id: " << canvas.id() << "\n";


## Create the context and start rendering

Run this cell after the canvas output has rendered. The browser transfers the
canvas to the worker asynchronously. `require_ready()` does not wait; it throws
a clear error until the transfer completes, so rerun this cell if needed.


In [ ]:
#include <canvas_host/canvas_host.hpp>

#include <GLES3/gl3.h>
#include <emscripten.h>
#include <emscripten/eventloop.h>

#include <iostream>
#include <string>

[]() {
    struct State
    {
        GLuint program = 0;
        GLuint vao = 0;
        GLint pointer_location = -1;
        GLint pressed_location = -1;
        int width = 0;
        int height = 0;
        float pointer_x = 0.5f;
        float pointer_y = 0.5f;
        float pressed = 0.0f;

        void render()
        {
            glViewport(0, 0, width, height);
            glClearColor(0.02f, 0.03f, 0.06f, 1.0f);
            glClear(GL_COLOR_BUFFER_BIT);

            glUseProgram(program);
            glUniform2f(pointer_location, pointer_x, pointer_y);
            glUniform1f(pressed_location, pressed);
            glBindVertexArray(vao);
            glDrawArrays(GL_TRIANGLES, 0, 3);
            glBindVertexArray(0);
        }

        void tick()
        {
            canvas_host::nl::json event;
            while (canvas.poll_event(event)) {
                const std::string type = event.value("type", std::string());
                std::cout << type << std::endl;

                if (type == "pointermove" || type == "pointerdown" || type == "pointerup") {
                    const double css_width = event.value("cssWidth", 1.0);
                    const double css_height = event.value("cssHeight", 1.0);
                    pointer_x = static_cast<float>(event.value("x", 0.0) / css_width);
                    pointer_y = 1.0f - static_cast<float>(event.value("y", 0.0) / css_height);
                }
                if (type == "pointerdown" || type == "keydown") {
                    pressed = 1.0f;
                } else if (type == "pointerup" || type == "pointercancel" || type == "keyup") {
                    pressed = 0.0f;
                }
            }
            render();
        }
    };

    // Clear a loop and context left by an earlier run of this cell.
    emscripten_run_script(
        "if (globalThis.__canvas_host_webgl_interval) {"
        "  clearInterval(globalThis.__canvas_host_webgl_interval);"
        "  globalThis.__canvas_host_webgl_interval = 0;"
        "}");
    const int previous_context =
        emscripten_run_script_int("globalThis.__canvas_host_webgl_context || 0");
    if (previous_context != 0) {
        canvas.destroy_webgl2_context(previous_context);
    }
    emscripten_run_script("globalThis.__canvas_host_webgl_context = 0;");

    canvas.require_ready();

    State* state = new State;
    state->width = canvas.pixel_width();
    state->height = canvas.pixel_height();

    const int context = canvas_host::create_webgl2_context(canvas);
    if (context != 0) {
        emscripten_run_script(
            ("globalThis.__canvas_host_webgl_context = " +
             std::to_string(context) + ";").c_str());
    }
    canvas.start_events();

    constexpr const char* vertex_shader = R"GLSL(#version 300 es
out vec2 v_uv;
void main()
{
    vec2 p = vec2(float((gl_VertexID << 1) & 2), float(gl_VertexID & 2));
    v_uv = p;
    gl_Position = vec4(p * 2.0 - 1.0, 0.0, 1.0);
}
)GLSL";

    constexpr const char* fragment_shader = R"GLSL(#version 300 es
precision highp float;
in vec2 v_uv;
uniform vec2 u_pointer;
uniform float u_pressed;
out vec4 frag_color;
void main()
{
    float glow = 1.0 - smoothstep(0.0, 0.35, distance(v_uv, u_pointer));
    vec3 base = mix(vec3(0.08, 0.25, 0.55), vec3(0.85, 0.35, 0.10), u_pressed);
    frag_color = vec4(base + glow, 1.0);
}
)GLSL";

    auto compile_shader = [](GLenum type, const char* source) -> GLuint {
        GLuint shader = glCreateShader(type);
        glShaderSource(shader, 1, &source, nullptr);
        glCompileShader(shader);

        GLint ok = GL_FALSE;
        glGetShaderiv(shader, GL_COMPILE_STATUS, &ok);
        if (ok == GL_FALSE) {
            char log[4096] = {};
            glGetShaderInfoLog(shader, static_cast<GLsizei>(sizeof(log)), nullptr, log);
            std::cout << "Shader compile error: " << log << "\n";
            glDeleteShader(shader);
            return 0;
        }
        return shader;
    };

    auto link_program = [](GLuint vertex, GLuint fragment) -> GLuint {
        GLuint program = glCreateProgram();
        glAttachShader(program, vertex);
        glAttachShader(program, fragment);
        glLinkProgram(program);

        GLint ok = GL_FALSE;
        glGetProgramiv(program, GL_LINK_STATUS, &ok);
        if (ok == GL_FALSE) {
            char log[4096] = {};
            glGetProgramInfoLog(program, static_cast<GLsizei>(sizeof(log)), nullptr, log);
            std::cout << "Program link error: " << log << "\n";
            glDeleteProgram(program);
            return 0;
        }
        return program;
    };

    if (context == 0) {
        std::cout << "WebGL2 context creation failed\n";
    } else {
        const GLuint vertex = compile_shader(GL_VERTEX_SHADER, vertex_shader);
        const GLuint fragment = compile_shader(GL_FRAGMENT_SHADER, fragment_shader);
        if (vertex != 0 && fragment != 0) {
            state->program = link_program(vertex, fragment);
        }
        glDeleteShader(vertex);
        glDeleteShader(fragment);

        if (state->program == 0) {
            std::cout << "Shader setup failed\n";
        } else {
            state->pointer_location = glGetUniformLocation(state->program, "u_pointer");
            state->pressed_location = glGetUniformLocation(state->program, "u_pressed");
            glGenVertexArrays(1, &state->vao);
            glBindVertexArray(state->vao);
            state->render();
            glBindVertexArray(0);

            auto on_tick = [](void* user) {
                static_cast<State*>(user)->tick();
            };
            const int interval = emscripten_set_interval(on_tick, 16, state);
            emscripten_run_script(
                ("globalThis.__canvas_host_webgl_interval = " +
                 std::to_string(interval) + ";").c_str());
            std::cout << "WebGL2 " << state->width << "x" << state->height
                      << " px; move the pointer, click, or type.\n";
        }
    }
}();


## Stop

Stop the render loop, destroy the WebGL2 context, and close the canvas-host
channels.


In [ ]:
#include <emscripten.h>

#include <iostream>

emscripten_run_script(
    "if (globalThis.__canvas_host_webgl_interval) {"
    "  clearInterval(globalThis.__canvas_host_webgl_interval);"
    "  globalThis.__canvas_host_webgl_interval = 0;"
    "}");

const int context =
    emscripten_run_script_int("globalThis.__canvas_host_webgl_context || 0");
if (context != 0) {
    canvas.destroy_webgl2_context(context);
}
emscripten_run_script("globalThis.__canvas_host_webgl_context = 0;");

canvas.stop();
std::cout << "canvas stopped\n";
